# Notebook 05 — Model Training & Validation

**Input:** `data/features.csv`  
**Outputs:** `models/elocoach_model.pkl`, `models/cohort_distributions.pkl`

**Scope steps covered:** 1 (feature matrix), 2 (LR baseline), 3 (XGBoost tuning), 4 (comparison + selection)

**Scope conflict — documented:**  
Scope calls for cohort SHAP at '1200 vs 1400 Elo'. Actual data spans 1484–1898 Elo —  
no games exist below 1484. Cohort boundaries will be redefined in `06_shap_analysis.ipynb`  
based on the actual Elo distribution (see DEC-009).

**AUC < 0.60 → STOP.** Diagnostic checklist in Section 6.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import pathlib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap

sns.set_theme(style='whitegrid')
pathlib.Path('../models').mkdir(exist_ok=True)
pathlib.Path('../data/plots').mkdir(exist_ok=True)

FEATURES_PATH = '../data/features.csv'
RANDOM_STATE  = 42

# Columns excluded from model feature matrix X (DEC-009, DEC-010)
EXCLUDE_FROM_X = [
    'result',            # label
    'elo',               # metadata — cohort reference only, not a tactical feature (DEC-009)
    'civilization_id',   # categorical, not yet encoded — defer to v2
    'duration_min',      # target leakage (DEC-010)
    'apm_castle_age',    # collinear proxy — redundant with move_count/order_count (DEC-010)
    'apm_imperial_age',  # collinear proxy — redundant with villagers/mil_trained (DEC-010)
]

print('Setup OK')
print(f'XGBoost version: {xgb.__version__}')

## Section 1 — Feature Matrix X and Target y

Confirm no leakage: `elo` must not be in X. Stratified 80/20 split preserves class balance.  
Elo is carried alongside the test set for cohort analysis in notebook 06.

In [ ]:
df = pd.read_csv(FEATURES_PATH)
print(f'Loaded: {df.shape}')
print(f'Class balance:\n{df["result"].value_counts().to_string()}')

y = df['result'].astype(int)
X = df.drop(columns=[c for c in EXCLUDE_FROM_X if c in df.columns])

print(f'\nFeature matrix X: {X.shape[1]} features x {X.shape[0]} rows')
print(f'Excluded: {[c for c in EXCLUDE_FROM_X if c in df.columns]}')
print(f'\nLeakage check — elo in X: {"elo" in X.columns}  (must be False)')
print(f'\nFeatures in X:')
for c in X.columns:
    null_n = X[c].isnull().sum()
    tag = f'  ({null_n} NaN — XGBoost routes natively)' if null_n > 0 else ''
    print(f'  {c}{tag}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Carry Elo alongside test set for notebook 06 cohort analysis
elo_test = df.loc[X_test.index, 'elo'].reset_index(drop=True)

print(f'Train: {X_train.shape[0]} rows  |  Test: {X_test.shape[0]} rows')
print(f'Train balance: {dict(y_train.value_counts())}')
print(f'Test balance:  {dict(y_test.value_counts())}')

## Section 2 — Logistic Regression Baseline

LR cannot handle NaN natively — median imputation applied inside a Pipeline  
to prevent test-set leakage. StandardScaler normalises feature magnitudes.  
This sets the performance floor XGBoost must beat.

In [ ]:
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('lr',      LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                   class_weight='balanced', C=1.0))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

auc_lr  = roc_auc_score(y_test, y_prob_lr)
f1_lr   = f1_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)

print('Logistic Regression Baseline')
print(f'  AUC-ROC:   {auc_lr:.4f}')
print(f'  F1:        {f1_lr:.4f}')
print(f'  Precision: {prec_lr:.4f}')
print(f'  Recall:    {rec_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Loss','Win']))

In [ ]:
print("""
Interpretation:
Logistic Regression treats each feature as having an independent, linear contribution
to win probability. AOE2 macro decisions are inherently interactive — a fast Feudal
time only predicts a win when combined with an aggressive military follow-up, and a
late Castle time only hurts if not compensated by economic advantage. These interactions
cannot be captured by a linear model. This AUC score therefore represents the lower
bound of what the feature set can achieve — if XGBoost cannot beat it, the feature set
itself lacks sufficient non-linear signal at this sample size.
""")

## Section 3 — XGBoost with RandomizedSearchCV

5-fold stratified CV, 20 iterations. NaN handled natively via `tree_method='hist'`.  
Hyperparameter ranges conservative for a ~280-row training set to limit overfitting.

In [ ]:
param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 4, 5, 6],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9],
    'min_child_weight':  [1, 3, 5, 10],
    'gamma':             [0, 0.1, 0.2, 0.5],
    'reg_alpha':         [0, 0.1, 0.5, 1.0],
    'reg_lambda':        [1, 1.5, 2, 5],
}

xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=RANDOM_STATE,
    tree_method='hist',
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print(f'Best CV AUC: {search.best_score_:.4f}')
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
best_xgb   = search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

auc_xgb  = roc_auc_score(y_test, y_prob_xgb)
f1_xgb   = f1_score(y_test, y_pred_xgb)
prec_xgb = precision_score(y_test, y_pred_xgb)
rec_xgb  = recall_score(y_test, y_pred_xgb)

print('XGBoost (tuned)')
print(f'  AUC-ROC:   {auc_xgb:.4f}')
print(f'  F1:        {f1_xgb:.4f}')
print(f'  Precision: {prec_xgb:.4f}')
print(f'  Recall:    {rec_xgb:.4f}')
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Loss','Win']))

## Section 4 — Model Comparison and Selection

ROC curves + confusion matrices side by side. Winner = higher test AUC.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
for label, y_prob in [('Logistic Regression', y_prob_lr), ('XGBoost', y_prob_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{label}  (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()

for ax, preds, probs, title in [
    (axes[1], y_pred_lr,  y_prob_lr,  'Logistic Regression'),
    (axes[2], y_pred_xgb, y_prob_xgb, 'XGBoost (tuned)'),
]:
    cm = confusion_matrix(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Loss','Win'], yticklabels=['Loss','Win'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{title}  (AUC={auc:.3f})')

plt.tight_layout()
plt.savefig('../data/plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

comparison = pd.DataFrame({
    'Model':     ['Logistic Regression', 'XGBoost (tuned)'],
    'AUC-ROC':   [round(auc_lr,4),   round(auc_xgb,4)],
    'F1':        [round(f1_lr,4),    round(f1_xgb,4)],
    'Precision': [round(prec_lr,4),  round(prec_xgb,4)],
    'Recall':    [round(rec_lr,4),   round(rec_xgb,4)],
})
print(comparison.to_string(index=False))

In [ ]:
winning_model = best_xgb if auc_xgb >= auc_lr else lr_pipeline
winning_name  = 'XGBoost' if auc_xgb >= auc_lr else 'Logistic Regression'
winning_auc   = max(auc_xgb, auc_lr)

print(f'Selected: {winning_name}  (AUC = {winning_auc:.4f})')
print()
if winning_name == 'XGBoost':
    margin = auc_xgb - auc_lr
    print(f'Selection rationale: XGBoost outperforms LR by {margin:.4f} AUC. AOE2 macro '
          f'decisions interact non-linearly — fast Feudal timing is only predictive of '
          f'wins when paired with aggressive military follow-up. Tree-based splits capture '
          f'these interactions natively. XGBoost also routes NaN values to the optimal '
          f'split direction, correctly encoding "never researched" in tech timing features.')
else:
    print('Selection rationale: LR matched or exceeded XGBoost. This suggests the '
          'win/loss signal is approximately linear at this sample size, or the dataset '
          'is too small for XGBoost to learn reliable non-linear interactions. '
          'Document and consider collecting more replays before proceeding.')

if winning_auc < 0.60:
    print('\nWARNING: AUC < 0.60 — go to Section 6 diagnostic checklist. Do not proceed to notebook 06.')

## Section 5 — Save Model and Cohort Distributions

Save winning model and per-feature percentile distributions for the recommendation  
engine in notebook 07.

In [ ]:
model_path = '../models/elocoach_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(winning_model, f)
print(f'Model saved → {model_path}')

X_full   = df.drop(columns=[c for c in EXCLUDE_FROM_X if c in df.columns])
elo_full = df['elo']

cohort_distributions = {
    'feature_cols':  list(X_full.columns),
    'elo_range':     (float(elo_full.min()), float(elo_full.max())),
    'elo_median':    float(elo_full.median()),
    'percentiles':   {},
    'winning_model': winning_name,
    'winning_auc':   winning_auc,
}

for col in X_full.columns:
    vals = X_full[col].dropna()
    cohort_distributions['percentiles'][col] = {
        p: float(np.percentile(vals, p)) for p in [10, 25, 50, 75, 90]
    }

dist_path = '../models/cohort_distributions.pkl'
with open(dist_path, 'wb') as f:
    pickle.dump(cohort_distributions, f)
print(f'Cohort distributions saved → {dist_path}')
print(f'Elo range: {elo_full.min():.0f}–{elo_full.max():.0f}  (median {elo_full.median():.0f})')
print('Cohort split boundaries to be defined in 06_shap_analysis.ipynb.')

## Section 6 — AUC < 0.60 Diagnostic Checklist

**Only run if winning model AUC < 0.60.** Check in order per scope documentation.

In [ ]:
if winning_auc >= 0.60:
    print(f'AUC = {winning_auc:.4f} — diagnostic not needed. Proceed to 06_shap_analysis.ipynb.')
else:
    print('AUC < 0.60 — running checklist...\n')

    print('CHECK 1 — Data leakage')
    suspects = ['elo', 'duration_min', 'resign_time_min', 'resigned']
    found = [c for c in suspects if c in X.columns]
    print(f'  Leakage suspects in X: {found if found else "None found — OK"}')

    print('\nCHECK 2 — Feature distributions')
    X_full_ = df.drop(columns=[c for c in EXCLUDE_FROM_X if c in df.columns])
    diffs = (X_full_[df['result']==1].mean() - X_full_[df['result']==0].mean()).abs().sort_values(ascending=False)
    print('  Top 5 mean differences (win vs loss):')
    print(diffs.head().to_string())

    print(f'\nCHECK 3 — Training data volume')
    print(f'  Training rows: {len(X_train)}')
    print(f'  If < 250 rows, download more replays before re-tuning.')

    print('\nCHECK 4 — Label noise')
    print('  Inspect parsed_replays.csv for:')
    print('  - Rage-quits (resign_time_min < 10 min)')
    print('  - Smurf indicators (Elo mismatch > 300 between players)')
    print('  Filter suspect rows and retrain.')
    print('\nLog all findings in ISSUES before escalating.')